In [6]:
import pandas as pd
import numpy as np

# Reto 1 — campanias_marketing.csv
np.random.seed(42)
months = pd.date_range(start='2023-01-01', periods=24, freq='MS').strftime('%Y-%m')
month_effect = np.tile([1.2, 1.0, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.0, 0.95, 0.9, 1.1], 2)

# Funciones de CAC con pendiente positiva (más usuarios = más caro unitariamente)
clientes_youtube = np.random.randint(90, 180, size=24)
clientes_googleads = np.random.randint(100, 190, size=24)

# CAC base + incremento por volumen de clientes + efecto estacional
cac_youtube = 15 + (0.35 * (clientes_youtube / 10) * month_effect) + np.random.normal(0, 0.5, size=len(months))
cac_googleads = 14 + (0.45 * (clientes_googleads / 10) * month_effect) + np.random.normal(0, 0.5, size=len(months))

pauta_youtube_usd = (clientes_youtube * cac_youtube).round(2)
pauta_googleads_usd = (clientes_googleads * cac_googleads).round(2)

campanias_marketing = pd.DataFrame({
    'mes': months,
    'pauta_youtube_usd': pauta_youtube_usd,
    'clientes_youtube': clientes_youtube,
    'pauta_googleads_usd': pauta_googleads_usd,
    'clientes_googleads': clientes_googleads
})

campanias_marketing.to_csv("data/panda_cosmos_marketing_expenses.csv")

In [14]:
# Reto 2 — solicitudes_servicio_panda.csv
# Create non-linear trend using exponential growth with dampening
x = np.arange(96)  # Changed to 96 months (8 years)
trend = 1200 + 800 * (1 - np.exp(-x/24)) + 100 * x**0.3

# Monthly seasonality factors (higher in busy months, lower in slow months)
# High: Dec(12), Nov(11), Jul(7), Jun(6) - holidays, summer
# Low: Jan(1), Feb(2), Aug(8) - post-holidays, vacation
monthly_factors = np.array([0.85, 0.8, 0.95, 1.0, 1.05, 1.2, 
                          1.15, 0.85, 1.0, 1.05, 1.25, 1.3])
seasonal_pattern = np.tile(monthly_factors, 8)  # Changed to 8 years

# Apply seasonality to trend
seasonality = trend * (seasonal_pattern - 1)

# Add some random noise
noise = np.random.normal(0, 50, 96)  # Changed to 96 months

# Combine components and ensure all values are positive
solicitudes = trend + seasonality + noise
solicitudes = np.maximum(solicitudes, 0)  # Ensure no negative values

solicitudes_servicio_panda = pd.DataFrame({
    'fecha': pd.date_range(start='2017-01-01', periods=96, freq='MS'),  # Changed to 96 periods
    'solicitudes': solicitudes.astype(int)
})

solicitudes_servicio_panda.to_csv("data/panda_cosmos_solicitudes_servicio.csv")

In [23]:
# Reto 3 — Historial de conversión e ingresos
n_users = 1000
converted = np.random.choice([1, 0], size=n_users, p=[0.35, 0.65])
revenue = np.where(
    converted == 1,
    np.random.normal(loc=950, scale=180, size=n_users),
    0
).round(2)

historial_conversion_ingresos = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'converted': converted,
    'revenue': revenue
})

historial_conversion_ingresos.to_csv("data/panda_cosmos_historial_conversion_e_ingresos.csv")

# Reto 3 — Resultados prueba A/B
n_users_ab = 1100
converted_a = np.random.choice([1, 0], size=n_users_ab, p=[0.37, 0.63])
converted_b = np.random.choice([1, 0], size=n_users_ab, p=[0.33, 0.67])

revenue_a = np.where(
    converted_a == 1,
    np.random.normal(loc=920, scale=150, size=n_users_ab),
    0
).round(2)

revenue_b = np.where(
    converted_b == 1,
    np.random.normal(loc=1200, scale=160, size=n_users_ab),
    0
).round(2)

resultado_ab = pd.DataFrame({
    'user_id': range(1001, 1001 + 2*n_users_ab),
    'group': ['A'] * n_users_ab + ['B'] * n_users_ab,
    'revenue': np.concatenate([revenue_a, revenue_b]),
    'converted': np.concatenate([converted_a, converted_b])
})

resultado_ab.to_csv("data/panda_cosmos_resultado_ab_conversion_e_ingresos.csv")

In [24]:
# Reto 4 — datos_operador_panda.csv
np.random.seed(42)

# Definir grupos de conductores (clusters esperados)
n = 300
grupos = np.random.choice(["eficiente", "promedio", "deficiente"], size=n, p=[0.3, 0.5, 0.2])

# Inicializar listas
tiempo_entrega, entregas, valoracion = [], [], []
latencia, incidencias, retencion = [], [], []

for g in grupos:
    if g == "eficiente":
        tiempo_entrega.append(np.random.normal(28, 2))
        entregas.append(np.random.poisson(130))
        valoracion.append(np.random.uniform(4.5, 5.0))
        latencia.append(np.random.normal(10, 2))
        incidencias.append(np.random.beta(1, 20))
        retencion.append(np.random.choice([1, 0], p=[0.95, 0.05]))
    elif g == "promedio":
        tiempo_entrega.append(np.random.normal(33, 3))
        entregas.append(np.random.poisson(100))
        valoracion.append(np.random.uniform(3.8, 4.5))
        latencia.append(np.random.normal(14, 3))
        incidencias.append(np.random.beta(2, 10))
        retencion.append(np.random.choice([1, 0], p=[0.8, 0.2]))
    else:  # deficiente
        tiempo_entrega.append(np.random.normal(38, 4))
        entregas.append(np.random.poisson(70))
        valoracion.append(np.random.uniform(2.5, 3.8))
        latencia.append(np.random.normal(18, 4))
        incidencias.append(np.random.beta(3, 6))
        retencion.append(np.random.choice([1, 0], p=[0.5, 0.5]))

datos_operador_panda = pd.DataFrame({
    'conductor_id': range(1, n + 1),
    'tiempo_promedio_entrega': np.round(tiempo_entrega, 1),
    'entregas_completadas': entregas,
    'valoracion_cliente': np.round(valoracion, 2),
    'latencia_respuesta_app': np.round(latencia, 1),
    'tasa_incidencias': np.round(incidencias, 3),
    'retencion_anual': retencion
})

datos_operador_panda.to_csv("data/panda_cosmos_operadores.csv")